# Lab 2: Computation Graphs and Forward Propagation

## 🎯 Learning Objectives

By the end of this lab, you will understand:
- What computation graphs are and why they matter
- How to build a Value class that tracks operations
- Forward propagation through computation graphs
- NumPy fundamentals for efficient numeric computation
- How to visualize computation graphs

**Why This Matters:** Every deep learning framework (PyTorch, TensorFlow) uses computation graphs to track operations. This enables automatic differentiation, which we'll implement in later labs.

**Note:** You can now use NumPy! You've earned it after understanding the internals in Lab 1.

**Setup:** Run this cell first to download helper functions

In [ ]:
# Download helper files from GitHub (for Colab users)
import sys
import os

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Download utils.py for visualization
    !wget -q https://raw.githubusercontent.com/hhe0u0/micrograd-lab/claude/micrograd-lab-exercises-V14m2/utils.py
    print("✓ Helper files downloaded!")
else:
    # Local Jupyter - files should already exist
    if os.path.exists('utils.py'):
        print("✓ Helper files found locally!")
    else:
        print("⚠️  utils.py not found. Make sure you're in the micrograd-lab directory.")

## Part 1: What is a Computation Graph?

### The Big Idea

When you compute `d = (a + b) * c`, you're actually creating a **graph** of operations:

```
  a    b        Inputs
   \  /
    (+)    →  e   Intermediate result
     |  \ 
     |   c      Another input
     \  /
      (*)   →  d   Final output
```

This graph records:
1. **What values** were involved (a, b, c, e, d)
2. **What operations** were performed (+, *)
3. **How they connect** (parent-child relationships)

### Why This Matters

In deep learning:
- **Forward pass:** Compute outputs from inputs (what we do now)
- **Backward pass:** Compute gradients from outputs to inputs (next lab)

The computation graph is the **recipe** that lets us do both!

### Example: A Simple Computation Graph

Let's trace through `f = (a + b) * c` step by step:

```python
a = 2.0
b = 3.0
c = 4.0

# Step 1: a + b = 5.0
e = a + b  # e remembers it came from a and b via +

# Step 2: e * c = 20.0
f = e * c  # f remembers it came from e and c via *
```

The graph looks like:
```
a=2.0 ──┐
        ├──(+)──> e=5.0 ──┐
b=3.0 ──┘                 ├──(*)──> f=20.0
        c=4.0 ────────────┘
```

### Topological Sort: A Real-World Example

Here's an interactive example showing how topological sort works using a morning routine.

**Try completing tasks in order - notice which tasks can be done in parallel!**

In [ ]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Topological Sort — Morning Routine</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: sans-serif; background: #fff; color: #1a1a1a; padding: 24px; max-width: 740px; margin: 0 auto; }
  h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
  .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
  .node-g { cursor: pointer; }
  .node-g rect { transition: fill .25s, stroke .25s; }
  .node-g text { pointer-events: none; }
  .legend { display: flex; gap: 18px; flex-wrap: wrap; margin: 14px 0 0; font-size: 12px; color: #666; align-items: center; }
  .leg-dot { width: 14px; height: 14px; border-radius: 4px; display: inline-block; flex-shrink: 0; }
  .info { margin-top: 14px; padding: 12px 16px; border-radius: 8px; border: 0.5px solid #ddd; background: #f7f7f5; font-size: 13px; line-height: 1.7; min-height: 50px; }
  .order-list { margin-top: 10px; display: flex; flex-wrap: wrap; gap: 6px; }
  .order-pill { padding: 3px 10px; border-radius: 20px; font-size: 12px; background: #E6F1FB; color: #0C447C; border: 0.5px solid #185FA5; }
  button.rst { margin-top: 10px; padding: 6px 16px; border-radius: 6px; border: 0.5px solid #ccc; background: transparent; color: #1a1a1a; font-size: 13px; cursor: pointer; }
  button.rst:hover { background: #f0f0ee; }
</style>
</head>
<body>
<h1>Topological Sort — Morning Routine</h1>
<p class="subtitle">Click any available (blue) task to complete it. Tasks with no arrows between them can happen at the same time!</p>

<svg width="100%" viewBox="0 0 680 420" id="dag">
<defs>
  <marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
    <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
  </marker>
</defs>
<g id="edges"></g>
<g id="nodes"></g>
</svg>

<div class="legend">
  <span><span class="leg-dot" style="background:#E6F1FB;border:1px solid #185FA5"></span> Available</span>
  <span><span class="leg-dot" style="background:#9FE1CB;border:1px solid #0F6E56"></span> Done</span>
  <span><span class="leg-dot" style="background:#D3D1C7;border:1px solid #5F5E5A"></span> Blocked</span>
</div>

<div class="info" id="info">Click any <b>available</b> (blue) task to complete it. Tasks with no arrows between them can happen at the same time!</div>
<div class="order-list" id="orderList"></div>
<button class="rst" onclick="reset()">↺ Reset morning</button>

<script>
const tasks = {
  wake:     { label:"Wake up",      emoji:"⏰", deps:[] },
  shower:   { label:"Shower",       emoji:"🚿", deps:["wake"] },
  brushT:   { label:"Brush teeth",  emoji:"🪥", deps:["wake"] },
  brewC:    { label:"Brew coffee",  emoji:"☕", deps:["wake"] },
  hair:     { label:"Do hair",      emoji:"💇", deps:["shower"] },
  skincare: { label:"Skin care",    emoji:"🧴", deps:["shower"] },
  drinkC:   { label:"Drink coffee", emoji:"🥤", deps:["brewC"] },
  dressU:   { label:"Get dressed",  emoji:"👔", deps:["hair","skincare","brushT"] },
  go:       { label:"Head out",     emoji:"🚪", deps:["dressU","drinkC"] },
};

const W=136, H=40;
const pos = {
  wake:     [340,  36],
  shower:   [170, 120],
  brushT:   [340, 120],
  brewC:    [510, 120],
  hair:     [130, 210],
  skincare: [270, 210],
  drinkC:   [510, 210],
  dressU:   [270, 300],
  go:       [340, 390],
};

const state = {};
let completionOrder = [];

function isAvailable(id){ return !state[id] && tasks[id].deps.every(d=>state[d]); }
function isDone(id){ return !!state[id]; }

function nodeColor(id){
  if(isDone(id))      return {fill:'#9FE1CB', stroke:'#0F6E56', text:'#085041'};
  if(isAvailable(id)) return {fill:'#E6F1FB', stroke:'#185FA5', text:'#0C447C'};
  return {fill:'#F1EFE8', stroke:'#B4B2A9', text:'#888780'};
}

function edgeColor(from,to){
  if(isDone(from)&&isDone(to)) return '#1D9E75';
  if(isDone(from)&&isAvailable(to)) return '#378ADD';
  return '#B4B2A9';
}

function drawEdges(){
  const g=document.getElementById('edges');
  g.innerHTML='';
  for(const [id,t] of Object.entries(tasks)){
    for(const dep of t.deps){
      const [x1,y1]=pos[dep], [x2,y2]=pos[id];
      const sy=y1+H/2+1, ey=y2-H/2-2;
      const my=(sy+ey)/2;
      const col=edgeColor(dep,id);
      const p=document.createElementNS('http://www.w3.org/2000/svg','path');
      p.setAttribute('d',`M${x1},${sy} C${x1},${my} ${x2},${my} ${x2},${ey}`);
      p.setAttribute('fill','none');
      p.setAttribute('stroke',col);
      p.setAttribute('stroke-width','1.5');
      p.setAttribute('marker-end','url(#arr)');
      g.appendChild(p);
    }
  }
}

function drawNodes(){
  const g=document.getElementById('nodes');
  g.innerHTML='';
  for(const [id,t] of Object.entries(tasks)){
    const [cx,cy]=pos[id];
    const {fill,stroke,text}=nodeColor(id);
    const avail=isAvailable(id);
    const ng=document.createElementNS('http://www.w3.org/2000/svg','g');
    ng.classList.add('node-g');
    ng.style.cursor=avail?'pointer':'default';
    if(avail) ng.onclick=()=>complete(id);

    const rect=document.createElementNS('http://www.w3.org/2000/svg','rect');
    rect.setAttribute('x',cx-W/2); rect.setAttribute('y',cy-H/2);
    rect.setAttribute('width',W); rect.setAttribute('height',H);
    rect.setAttribute('rx','8'); rect.setAttribute('stroke-width','1.5');
    rect.setAttribute('fill',fill); rect.setAttribute('stroke',stroke);
    ng.appendChild(rect);

    const em=document.createElementNS('http://www.w3.org/2000/svg','text');
    em.setAttribute('x',cx-W/2+18); em.setAttribute('y',cy);
    em.setAttribute('text-anchor','middle'); em.setAttribute('dominant-baseline','central');
    em.style.fontSize='15px';
    em.textContent=isDone(id)?'✓':t.emoji;
    ng.appendChild(em);

    const lab=document.createElementNS('http://www.w3.org/2000/svg','text');
    lab.setAttribute('x',cx+12); lab.setAttribute('y',cy);
    lab.setAttribute('text-anchor','middle'); lab.setAttribute('dominant-baseline','central');
    lab.setAttribute('fill',text);
    lab.style.fontSize='12px'; lab.style.fontWeight='500';
    lab.textContent=t.label;
    ng.appendChild(lab);

    g.appendChild(ng);
  }
}

function render(){ drawEdges(); drawNodes(); }

function complete(id){
  state[id]=true;
  completionOrder.push(id);
  const newUnlocked=Object.keys(tasks).filter(k=>!state[k]&&isAvailable(k)).map(k=>tasks[k].label);
  render();
  let msg=`<b>✓ ${tasks[id].emoji} ${tasks[id].label}</b> done!`;
  if(newUnlocked.length) msg+=`<br><span style="color:#185FA5">🔓 Now unlocked: ${newUnlocked.join(', ')}</span>`;
  document.getElementById('info').innerHTML=msg;
  updateOrderList();
  if(Object.keys(tasks).every(k=>state[k])){
    document.getElementById('info').innerHTML='<b>🎉 Morning done!</b> Notice how shower unlocks two parallel branches — that\'s the power of spotting independent nodes in a topo sort.';
  }
}

function updateOrderList(){
  document.getElementById('orderList').innerHTML=
    '<span style="font-size:12px;color:#666;align-self:center">Your order: </span>'+
    completionOrder.map((id,i)=>`<span class="order-pill">${i+1}. ${tasks[id].label}</span>`).join('');
}

function reset(){
  for(const k of Object.keys(tasks)) delete state[k];
  completionOrder=[];
  document.getElementById('orderList').innerHTML='';
  document.getElementById('info').innerHTML='Click any <b>available</b> (blue) task to complete it. Tasks with no arrows between them can happen at the same time!';
  render();
}

render();
</script>
</body>
</html>


In [ ]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Topological Sort — Morning Routine</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: sans-serif; background: #fff; color: #1a1a1a; padding: 24px; max-width: 740px; margin: 0 auto; }
  h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
  .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
  .node-g { cursor: pointer; }
  .node-g rect { transition: fill .25s, stroke .25s; }
  .node-g text { pointer-events: none; }
  .legend { display: flex; gap: 18px; flex-wrap: wrap; margin: 14px 0 0; font-size: 12px; color: #666; align-items: center; }
  .leg-dot { width: 14px; height: 14px; border-radius: 4px; display: inline-block; flex-shrink: 0; }
  .info { margin-top: 14px; padding: 12px 16px; border-radius: 8px; border: 0.5px solid #ddd; background: #f7f7f5; font-size: 13px; line-height: 1.7; min-height: 50px; }
  .order-list { margin-top: 10px; display: flex; flex-wrap: wrap; gap: 6px; }
  .order-pill { padding: 3px 10px; border-radius: 20px; font-size: 12px; background: #E6F1FB; color: #0C447C; border: 0.5px solid #185FA5; }
  button.rst { margin-top: 10px; padding: 6px 16px; border-radius: 6px; border: 0.5px solid #ccc; background: transparent; color: #1a1a1a; font-size: 13px; cursor: pointer; }
  button.rst:hover { background: #f0f0ee; }
</style>
</head>
<body>
<h1>Topological Sort — Morning Routine</h1>
<p class="subtitle">Click any available (blue) task to complete it. Tasks with no arrows between them can happen at the same time!</p>

<svg width="100%" viewBox="0 0 680 420" id="dag">
<defs>
  <marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
    <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
  </marker>
</defs>
<g id="edges"></g>
<g id="nodes"></g>
</svg>

<div class="legend">
  <span><span class="leg-dot" style="background:#E6F1FB;border:1px solid #185FA5"></span> Available</span>
  <span><span class="leg-dot" style="background:#9FE1CB;border:1px solid #0F6E56"></span> Done</span>
  <span><span class="leg-dot" style="background:#D3D1C7;border:1px solid #5F5E5A"></span> Blocked</span>
</div>

<div class="info" id="info">Click any <b>available</b> (blue) task to complete it. Tasks with no arrows between them can happen at the same time!</div>
<div class="order-list" id="orderList"></div>
<button class="rst" onclick="reset()">↺ Reset morning</button>

<script>
const tasks = {
  wake:     { label:"Wake up",      emoji:"⏰", deps:[] },
  shower:   { label:"Shower",       emoji:"🚿", deps:["wake"] },
  brushT:   { label:"Brush teeth",  emoji:"🪥", deps:["wake"] },
  brewC:    { label:"Brew coffee",  emoji:"☕", deps:["wake"] },
  hair:     { label:"Do hair",      emoji:"💇", deps:["shower"] },
  skincare: { label:"Skin care",    emoji:"🧴", deps:["shower"] },
  drinkC:   { label:"Drink coffee", emoji:"🥤", deps:["brewC"] },
  dressU:   { label:"Get dressed",  emoji:"👔", deps:["hair","skincare","brushT"] },
  go:       { label:"Head out",     emoji:"🚪", deps:["dressU","drinkC"] },
};

const W=136, H=40;
const pos = {
  wake:     [340,  36],
  shower:   [170, 120],
  brushT:   [340, 120],
  brewC:    [510, 120],
  hair:     [130, 210],
  skincare: [270, 210],
  drinkC:   [510, 210],
  dressU:   [270, 300],
  go:       [340, 390],
};

const state = {};
let completionOrder = [];

function isAvailable(id){ return !state[id] && tasks[id].deps.every(d=>state[d]); }
function isDone(id){ return !!state[id]; }

function nodeColor(id){
  if(isDone(id))      return {fill:'#9FE1CB', stroke:'#0F6E56', text:'#085041'};
  if(isAvailable(id)) return {fill:'#E6F1FB', stroke:'#185FA5', text:'#0C447C'};
  return {fill:'#F1EFE8', stroke:'#B4B2A9', text:'#888780'};
}

function edgeColor(from,to){
  if(isDone(from)&&isDone(to)) return '#1D9E75';
  if(isDone(from)&&isAvailable(to)) return '#378ADD';
  return '#B4B2A9';
}

function drawEdges(){
  const g=document.getElementById('edges');
  g.innerHTML='';
  for(const [id,t] of Object.entries(tasks)){
    for(const dep of t.deps){
      const [x1,y1]=pos[dep], [x2,y2]=pos[id];
      const sy=y1+H/2+1, ey=y2-H/2-2;
      const my=(sy+ey)/2;
      const col=edgeColor(dep,id);
      const p=document.createElementNS('http://www.w3.org/2000/svg','path');
      p.setAttribute('d',`M${x1},${sy} C${x1},${my} ${x2},${my} ${x2},${ey}`);
      p.setAttribute('fill','none');
      p.setAttribute('stroke',col);
      p.setAttribute('stroke-width','1.5');
      p.setAttribute('marker-end','url(#arr)');
      g.appendChild(p);
    }
  }
}

function drawNodes(){
  const g=document.getElementById('nodes');
  g.innerHTML='';
  for(const [id,t] of Object.entries(tasks)){
    const [cx,cy]=pos[id];
    const {fill,stroke,text}=nodeColor(id);
    const avail=isAvailable(id);
    const ng=document.createElementNS('http://www.w3.org/2000/svg','g');
    ng.classList.add('node-g');
    ng.style.cursor=avail?'pointer':'default';
    if(avail) ng.onclick=()=>complete(id);

    const rect=document.createElementNS('http://www.w3.org/2000/svg','rect');
    rect.setAttribute('x',cx-W/2); rect.setAttribute('y',cy-H/2);
    rect.setAttribute('width',W); rect.setAttribute('height',H);
    rect.setAttribute('rx','8'); rect.setAttribute('stroke-width','1.5');
    rect.setAttribute('fill',fill); rect.setAttribute('stroke',stroke);
    ng.appendChild(rect);

    const em=document.createElementNS('http://www.w3.org/2000/svg','text');
    em.setAttribute('x',cx-W/2+18); em.setAttribute('y',cy);
    em.setAttribute('text-anchor','middle'); em.setAttribute('dominant-baseline','central');
    em.style.fontSize='15px';
    em.textContent=isDone(id)?'✓':t.emoji;
    ng.appendChild(em);

    const lab=document.createElementNS('http://www.w3.org/2000/svg','text');
    lab.setAttribute('x',cx+12); lab.setAttribute('y',cy);
    lab.setAttribute('text-anchor','middle'); lab.setAttribute('dominant-baseline','central');
    lab.setAttribute('fill',text);
    lab.style.fontSize='12px'; lab.style.fontWeight='500';
    lab.textContent=t.label;
    ng.appendChild(lab);

    g.appendChild(ng);
  }
}

function render(){ drawEdges(); drawNodes(); }

function complete(id){
  state[id]=true;
  completionOrder.push(id);
  const newUnlocked=Object.keys(tasks).filter(k=>!state[k]&&isAvailable(k)).map(k=>tasks[k].label);
  render();
  let msg=`<b>✓ ${tasks[id].emoji} ${tasks[id].label}</b> done!`;
  if(newUnlocked.length) msg+=`<br><span style="color:#185FA5">🔓 Now unlocked: ${newUnlocked.join(', ')}</span>`;
  document.getElementById('info').innerHTML=msg;
  updateOrderList();
  if(Object.keys(tasks).every(k=>state[k])){
    document.getElementById('info').innerHTML='<b>🎉 Morning done!</b> Notice how shower unlocks two parallel branches — that\'s the power of spotting independent nodes in a topo sort.';
  }
}

function updateOrderList(){
  document.getElementById('orderList').innerHTML=
    '<span style="font-size:12px;color:#666;align-self:center">Your order: </span>'+
    completionOrder.map((id,i)=>`<span class="order-pill">${i+1}. ${tasks[id].label}</span>`).join('');
}

function reset(){
  for(const k of Object.keys(tasks)) delete state[k];
  completionOrder=[];
  document.getElementById('orderList').innerHTML='';
  document.getElementById('info').innerHTML='Click any <b>available</b> (blue) task to complete it. Tasks with no arrows between them can happen at the same time!';
  render();
}

render();
</script>
</body>
</html>


## Part 2: Building the Value Class

### Design Goals

Our `Value` class needs to:
1. **Store the data** (the actual number; for example `e=5.0`)
2. **Remember parents** (which Values it came from; for example, parents of `e` are `a` and `b`)
3. **Remember operation** (what operation created it; for example, the operator for `e` is `+`)
4. **Support operators** (`+`, `-`, `*`, `/`, `sigmoid`, etc)

### The Value Class Structure

```python
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)        # The actual number
        self.grad = 0.0                 # Gradient (used in backward pass)
        self._prev = set(_children)     # Parent nodes in the graph
        self._op = _op                  # Operation that created this node
        self.label = label              # Optional name for visualization
```

### Demo: How It Works

In [ ]:
# Simple demo of the Value class
class ValueDemo:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, ValueDemo) else ValueDemo(other)
        out = ValueDemo(self.data + other.data, (self, other), '+')
        return out
    
    def __repr__(self):
        return f"Value(data={self.data:.4f})"

# Test it
a = ValueDemo(2.0, label='a')
b = ValueDemo(3.0, label='b')
c = a + b
c.label = 'c'

print(f"a = {a}")
print(f"b = {b}")
print(f"c = a + b = {c}")
print(f"\nc._op = '{c._op}'  (operation that created c)")
print(f"c._prev = {c._prev}  (parents of c)")
print(f"\n✓ The computation graph is being recorded!")

## Exercise 1: Implement the Value Class

### Your Task

Implement a complete `Value` class that:
1. Tracks data and computation history
2. Supports arithmetic operators (+, -, *, /)
3. Handles operations with plain numbers (e.g., `Value(5) + 3`)

### Starter Code

In [ ]:
class Value:
    """
    A Value stores a scalar and tracks its computation history.
    """
    
    def __init__(self, data, _children=(), _op='', label=''):
        """
        Args:
            data: The scalar value (will be converted to float)
            _children: Tuple of parent Value nodes
            _op: String describing the operation ('+', '*', etc.)
            label: Optional name for this node (for visualization)
        """
        raise NotImplementedError("Implement __init__")
    
    def __add__(self, other):
        """
        Addition: self + other
        
        Must work with both Value objects and plain numbers:
        - Value(5) + Value(3) → Value(8)
        - Value(5) + 3 → Value(8)
        """
        #       data = self.data + other.data
        #       _children = (self, other)
        #       _op = '+'
        raise NotImplementedError("Implement __add__")
    
    def __mul__(self, other):
        """
        Multiplication: self * other
        """
        raise NotImplementedError("Implement __mul__")
    
    def __neg__(self):
        """Negation: -self"""
        return self * -1
    
    def __sub__(self, other):
        """Subtraction: self - other = self + (-other)"""
        return self + (-other)
    
    def __truediv__(self, other):
        """Division: self / other = self * (1/other)"""
        return self * (other ** -1)
    
    def __pow__(self, other):
        """
        Power: self ** other
        Note: other must be a number (int or float), not a Value
        """
        raise NotImplementedError("Implement __pow__")
    
    def __radd__(self, other):
        """Right addition: other + self (when other is not a Value)"""
        return self + other
    
    def __rmul__(self, other):
        """Right multiplication: other * self"""
        return self * other
    
    def __repr__(self):
        """String representation for printing."""
        return f"Value(data={self.data:.4f})"

### Test Your Implementation

In [ ]:
print("=" * 60)
print("Test 1: Basic Operations")
print("=" * 60)

a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = a + b
c.label = 'c'

print(f"a = {a}")
print(f"b = {b}")
print(f"c = a + b = {c}")
assert c.data == 5.0, f"Expected 5.0, got {c.data}"
assert c._op == '+', f"Expected '+', got {c._op}"
assert a in c._prev and b in c._prev, "c should have a and b as parents"
print("✓ Addition: PASS\n")

In [ ]:
print("=" * 60)
print("Test 2: Complex Expression")
print("=" * 60)

# f = (a + b) * c
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(4.0, label='c')

e = a + b
e.label = 'e'
f = e * c
f.label = 'f'

print(f"f = (a + b) * c = ({a.data} + {b.data}) * {c.data} = {f.data}")
assert f.data == 20.0, f"Expected 20.0, got {f.data}"
print("✓ Complex expression: PASS\n")

In [ ]:
print("=" * 60)
print("Test 3: Operations with Plain Numbers")
print("=" * 60)

x = Value(5.0)
y = x + 3  # Should work even though 3 is not a Value
z = 2 * x  # Should work with reversed operands

print(f"x = {x}")
print(f"x + 3 = {y}")
print(f"2 * x = {z}")

assert y.data == 8.0, f"Expected 8.0, got {y.data}"
assert z.data == 10.0, f"Expected 10.0, got {z.data}"
print("✓ Mixed operations: PASS\n")

In [ ]:
print("=" * 60)
print("Test 4: All Operators")
print("=" * 60)

a = Value(10.0)
b = Value(3.0)

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}")
print(f"a / b = {a / b}")
print(f"a ** 2 = {a ** 2}")

assert (a + b).data == 13.0
assert (a - b).data == 7.0
assert (a * b).data == 30.0
assert abs((a / b).data - 3.333333) < 0.001
assert (a ** 2).data == 100.0

print("\n✓ All operators: PASS\n")

In [ ]:
print("=" * 60)
print("🎉 All Tests Passed!")
print("=" * 60)
print("\nYour Value class can track computation graphs!")

## Part 3: Visualizing Computation Graphs

### Why Visualization Matters

Seeing the computation graph helps you:
- Understand how operations connect
- Debug issues in complex expressions
- Verify your implementation is correct

### Using the Visualization Helper

We provide a `visualize_graph` function (from Karpathy's micrograd) that draws the computation graph using Graphviz.

In [ ]:
from utils import visualize_graph

# Build a computation graph
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a * b
e.label = 'e'
d = e + c
d.label = 'd'

# Visualize it!
visualize_graph(d, title="Computation Graph: d = (a * b) + c")

**What you should see:**
- Nodes showing values (a=2.0, b=-3.0, c=10.0, e=-6.0, d=4.0)
- Operations (* and +) connecting them
- Arrows showing the flow from inputs to output
- Gradients are all 0.0 (we haven't computed them yet)

**Try visualizing your own expressions!**

In [ ]:
# Try a more complex expression
x = Value(2.0, label='x')
y = Value(3.0, label='y')
z = x ** 2 + y ** 2
z.label = 'z'

visualize_graph(z, title="Computation Graph: z = x² + y²")

## Part 4: Gradients and Backward Propagation

### What Are Gradients?

A **gradient** tells us **how much the output changes** when we nudge an input slightly.

Think of it as **sensitivity**:
- If `input.grad = 5`, then increasing input by 0.001 increases output by ≈0.005
- Large gradient = output is very sensitive to this input
- Small gradient = output barely cares about this input

### Why Gradients Matter for Deep Learning

In neural networks:
- **Forward pass:** Compute predictions from inputs
- **Backward pass:** Compute gradients to know which direction to adjust weights
- **Optimization:** Use gradients to update weights and reduce error

**The computation graph we built enables automatic gradient calculation!**

### Simple Example: Linear Function

For `f(x) = 2x`, the gradient is always 2:

```
f(3) = 6
f(3.001) = 6.002
Change = 6.002 - 6.000 = 0.002 = 2 × 0.001
Gradient = 0.002 / 0.001 = 2 ✓
```

In [ ]:
# Demo: Computing gradient numerically
def f(x):
    return 2 * x

x = 3.0
h = 0.001  # Small nudge

# Numerical gradient
f_x = f(x)
f_x_plus_h = f(x + h)
gradient = (f_x_plus_h - f_x) / h

print("=" * 60)
print("Numerical Gradient Demo")
print("=" * 60)
print(f"f(x) = 2x")
print(f"f({x}) = {f_x}")
print(f"f({x + h}) = {f_x_plus_h}")
print(f"Gradient = (f(x+h) - f(x)) / h = {gradient:.6f}")
print(f"Expected: 2.0 ✓")

### Gradients in Computation Graphs

For complex expressions like `f = (a + b) * c`, we need gradients of f with respect to each input:
- `df/da` = How does f change when a changes?
- `df/db` = How does f change when b changes?
- `df/dc` = How does f change when c changes?

**The Chain Rule** lets us compute these by traversing the computation graph backwards!

### Example: Computing Gradients

For `f = (a + b) * c` where a=2, b=3, c=4:

**Forward pass:**
```
e = a + b = 5
f = e * c = 20
```

**Backward pass (gradients):**
```
df/df = 1  (by definition)
df/dc = e = 5  (because f = e * c, so ∂(e*c)/∂c = e)
df/de = c = 4  (because f = e * c, so ∂(e*c)/∂e = c)
df/da = df/de * de/da = 4 * 1 = 4  (chain rule!)
df/db = df/de * de/db = 4 * 1 = 4  (chain rule!)
```

Let's verify this numerically:

In [ ]:
# Verify gradients numerically
def compute_f(a_val, b_val, c_val):
    e = a_val + b_val
    f = e * c_val
    return f

a, b, c = 2.0, 3.0, 4.0
h = 0.0001

f_original = compute_f(a, b, c)

# Gradient with respect to a
f_a_nudged = compute_f(a + h, b, c)
grad_a = (f_a_nudged - f_original) / h

# Gradient with respect to b
f_b_nudged = compute_f(a, b + h, c)
grad_b = (f_b_nudged - f_original) / h

# Gradient with respect to c
f_c_nudged = compute_f(a, b, c + h)
grad_c = (f_c_nudged - f_original) / h

print("=" * 60)
print("Numerical Gradient Verification")
print("=" * 60)
print(f"f = (a + b) * c = ({a} + {b}) * {c} = {f_original}")
print(f"\nNumerical gradients:")
print(f"df/da ≈ {grad_a:.4f}  (expected: 4.0)")
print(f"df/db ≈ {grad_b:.4f}  (expected: 4.0)")
print(f"df/dc ≈ {grad_c:.4f}  (expected: 5.0)")
print(f"\n✓ Matches our analytical calculation!")

### How to Compute Gradients: The Chain Rule

For complex expressions like `f = (a + b) * c`, we need gradients of f with respect to each input:
- `df/da` = How does f change when a changes?
- `df/db` = How does f change when b changes?
- `df/dc` = How does f change when c changes?

**The Chain Rule** lets us compute these by traversing the computation graph backwards!

### Gradient Rules for Basic Operations

For each operation, we need to know its **local gradient**:

**Addition: `out = a + b`**
- `dout/da = 1` (output increases by same amount as a)
- `dout/db = 1` (output increases by same amount as b)

**Multiplication: `out = a * b`**
- `dout/da = b` (output increases by b times the change in a)
- `dout/db = a` (output increases by a times the change in b)

**Power: `out = a ** n`**
- `dout/da = n * a^(n-1)` (power rule from calculus)

### Example: Computing Gradients Step-by-Step

For `f = (a + b) * c` where a=2, b=3, c=4:

**Forward pass:**
```
e = a + b = 5
f = e * c = 20
```

**Backward pass (work from output to inputs):**
```
df/df = 1  (by definition)
df/dc = e = 5  (because f = e * c, so ∂(e*c)/∂c = e)
df/de = c = 4  (because f = e * c, so ∂(e*c)/∂e = c)
df/da = df/de * de/da = 4 * 1 = 4  (chain rule!)
df/db = df/de * de/db = 4 * 1 = 4  (chain rule!)
```

Let's verify this numerically:

In [ ]:
class DemoValue:
    """Value class with backward pass for addition (demo)."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None  # Default: do nothing
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, DemoValue) else DemoValue(other)
        out = DemoValue(self.data + other.data, (self, other), '+')
        
        def _backward():
            # Gradient of addition:
            # If out = self + other, then:
            # - dout/dself = 1 (output increases by 1 when self increases by 1)
            # - dout/dother = 1
            # Chain rule: self.grad += out.grad * (dout/dself)
            self.grad += out.grad * 1.0  # or just: self.grad += out.grad
            other.grad += out.grad * 1.0  # or just: other.grad += out.grad
        
        out._backward = _backward  # Store the backward function
        return out
    
    def backward(self):
        """Compute all gradients using topological sort."""
        # Build topological order
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # Initialize output gradient
        self.grad = 1.0
        
        # Go backwards and call each _backward()
        for node in reversed(topo):
            node._backward()
    
    def __repr__(self):
        return f"DemoValue(data={self.data:.4f}, grad={self.grad:.4f})"

# Test it!
a = DemoValue(2.0, label='a')
b = DemoValue(3.0, label='b')
c = a + b
c.label = 'c'

print("Before backward():")
print(f"a.grad = {a.grad}")
print(f"b.grad = {b.grad}")

c.backward()  # Compute gradients!

print("\nAfter backward():")
print(f"a.grad = {a.grad}  (expected: 1.0)")
print(f"b.grad = {b.grad}  (expected: 1.0)")
print("\n✓ Addition backward pass works!")

## Exercise 2: Implement Backward Pass for All Operators

### Your Task

Now implement the `_backward()` function for multiplication and power operations.

**Gradient formulas you need:**

**Multiplication:** `out = a * b`
- `dout/da = b` (if b=3, changing a by 1 changes out by 3)
- `dout/db = a` (if a=2, changing b by 1 changes out by 2)

**Power:** `out = a ** n`
- `dout/da = n * a^(n-1)` (power rule from calculus)
  - Example: d(x²)/dx = 2x

**Chain rule:** `input.grad += out.grad * (dout/dinput)`

### Starter Code

In [ ]:
class Value:
    """Value class with backward pass - implement for all operators."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # Addition: both get same gradient
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            pass  # Remove this and add your implementation
        out._backward = _backward
        
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            pass  # Remove this and add your implementation
        out._backward = _backward
        
        return out
    
    def relu(self):
        """ReLU activation: max(0, self)"""
        out = Value(max(0.0, self.data), (self,), 'ReLU')
        
        def _backward():
            pass
        out._backward = _backward
        
        return out
    
    def backward(self):
        """Compute all gradients using topological sort."""
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1)
    def __radd__(self, other): return self + other
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

### Test Your Backward Implementation

In [ ]:
# Test multiplication backward
print("=" * 60)
print("Test 1: Multiplication Backward")
print("=" * 60)

a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = a * b
c.label = 'c'

c.backward()

print(f"c = a * b = {c.data}")
print(f"a.grad = {a.grad}  (expected: 3.0 = b.data)")
print(f"b.grad = {b.grad}  (expected: 2.0 = a.data)")

# Verify
assert abs(a.grad - 3.0) < 0.001, f"Expected a.grad=3.0, got {a.grad}"
assert abs(b.grad - 2.0) < 0.001, f"Expected b.grad=2.0, got {b.grad}"
print("✓ Multiplication backward: PASS\n")

In [ ]:
# Test power backward
print("=" * 60)
print("Test 2: Power Backward")
print("=" * 60)

x = Value(3.0, label='x')
y = x ** 2
y.label = 'y'

y.backward()

print(f"y = x^2 = {y.data}")
print(f"x.grad = {x.grad}  (expected: 6.0 = 2*x)")

# Verify with numerical gradient
h = 0.0001
numerical_grad = ((3.0 + h)**2 - 3.0**2) / h
print(f"Numerical verification: {numerical_grad:.4f}")

assert abs(x.grad - 6.0) < 0.001, f"Expected x.grad=6.0, got {x.grad}"
print("✓ Power backward: PASS\n")

In [ ]:
# Test complex expression
print("=" * 60)
print("Test 3: Complex Expression")
print("=" * 60)

# f = (a + b) * c
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(4.0, label='c')

e = a + b
e.label = 'e'
f = e * c
f.label = 'f'

f.backward()

print(f"f = (a + b) * c = {f.data}")
print(f"\nGradients:")
print(f"a.grad = {a.grad}  (expected: 4.0 = c.data)")
print(f"b.grad = {b.grad}  (expected: 4.0 = c.data)")
print(f"c.grad = {c.grad}  (expected: 5.0 = e.data)")

# Verify
assert abs(a.grad - 4.0) < 0.001
assert abs(b.grad - 4.0) < 0.001
assert abs(c.grad - 5.0) < 0.001
print("\n✓ Complex expression backward: PASS\n")

print("🎉 All backward passes work! You've implemented autograd!")

## Part 5: Introduction to NumPy

Now that you understand gradients and autograd, let's learn NumPy for efficient computation!

In [ ]:
import numpy as np

print("=" * 60)
print("Creating NumPy Arrays")
print("=" * 60)

# From Python list
a = np.array([1, 2, 3, 4])
print(f"1D array: {a}")
print(f"Shape: {a.shape}")
print(f"Type: {a.dtype}\n")

# 2D array (matrix)
b = np.array([[1, 2, 3], [4, 5, 6]])
print(f"2D array:\n{b}")
print(f"Shape: {b.shape}  (2 rows, 3 columns)\n")

# Special arrays
zeros = np.zeros((2, 3))  # All zeros
ones = np.ones((2, 3))    # All ones
identity = np.eye(3)       # Identity matrix

print(f"Zeros (2x3):\n{zeros}\n")
print(f"Ones (2x3):\n{ones}\n")
print(f"Identity (3x3):\n{identity}")

### Indexing and Slicing

In [ ]:
print("=" * 60)
print("Indexing and Slicing")
print("=" * 60)

x = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

print(f"Original array:\n{x}\n")

# Single element
print(f"x[0, 0] = {x[0, 0]}  (first element)")
print(f"x[1, 2] = {x[1, 2]}  (row 1, col 2)\n")

# Slicing rows
print(f"First row: x[0, :] = {x[0, :]}")
print(f"Last row: x[-1, :] = {x[-1, :]}\n")

# Slicing columns
print(f"First column: x[:, 0] = {x[:, 0]}")
print(f"Last column: x[:, -1] = {x[:, -1]}\n")

# Subarray
print(f"First 2 rows, first 3 cols:\n{x[:2, :3]}")

### Element-wise Operations

In [ ]:
print("=" * 60)
print("Element-wise Operations")
print("=" * 60)

a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print(f"a = {a}")
print(f"b = {b}\n")

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}  (element-wise!)")
print(f"a / b = {a / b}")
print(f"a ** 2 = {a ** 2}")

# Operations with scalars (broadcasting!)
print(f"\na + 10 = {a + 10}  (adds 10 to each element)")
print(f"a * 2 = {a * 2}  (multiplies each element by 2)")

### Matrix Multiplication in NumPy

NumPy provides two ways to do matrix multiplication:
1. `np.matmul(A, B)` or `A @ B` (recommended)
2. `np.dot(A, B)` (older, but common)

**Documentation:**
- `np.matmul`: https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
- `np.dot`: https://numpy.org/doc/stable/reference/generated/numpy.dot.html

## Summary

### What You've Learned

✅ **Computation graphs** track operations and their connections

✅ **Value class** records computation history for each operation

✅ **Forward propagation** computes outputs from inputs through the graph

✅ **Visualization** helps understand and debug computation graphs

✅ **Gradients** measure how sensitive outputs are to input changes

✅ **Numerical gradients** verify our understanding by nudging inputs

✅ **Chain rule** connects gradients through nested operations

✅ **Topological sort** enables reverse-mode automatic differentiation

✅ **Backward propagation** automatically computes all gradients

✅ **NumPy fundamentals** for efficient array operations

✅ **Matrix multiplication** with `@` operator and `np.matmul`

✅ **Linear regression with gradients** - combining Value objects with NumPy data

### Why This Matters

You've built a complete automatic differentiation system:
- ✓ Forward pass records operations
- ✓ Backward pass computes gradients automatically
- ✓ Value objects can be used with regular floats/NumPy
- ✓ Gradients have the same shape as parameters (shape [3] for 3 weights)

**This is exactly how PyTorch and TensorFlow work!**

### Next Steps

In **Lab 3**, you'll:
- Build neural network components (Neuron, Layer, MLP)
- Train a complete neural network on 2D data
- Visualize decision boundaries
- Solve the same problem as [Karpathy's micrograd demo](https://github.com/karpathy/micrograd/blob/master/demo.ipynb)

**Great work!** 🎉 You now have a working autograd engine!

In [ ]:
from utils import topological_sort

# Demo: Topological sort builds the correct order for backprop
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(4.0, label='c')

e = a + b
e.label = 'e'
f = e * c
f.label = 'f'

# Get nodes in topological order (output to inputs)
topo = topological_sort(f)

print("Topological order (for backward pass):")
for node in topo:
    print(f"  {node.label if node.label else 'unnamed'}: data={node.data}")

print("\n✓ This is the order we traverse when computing gradients!")

In [ ]:
# Complete Value class with backward pass (from micrograd)
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None  # Default: no gradient computation
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # Gradient of addition: both inputs get the same gradient
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            # Gradient of multiplication: each input gets other's value times output gradient
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), \"only supporting int/float powers\"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            # Gradient of power: other * x^(other-1) * out.grad
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        
        return out
    
    def backward(self):
        \"\"\"Compute gradients using reverse-mode automatic differentiation.\"\"\"
        # Build topological order
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        # Go backwards through graph
        self.grad = 1.0  # Gradient of output with respect to itself is 1
        for node in reversed(topo):
            node._backward()
    
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1)
    def __radd__(self, other): return self + other
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    
    def __repr__(self):
        return f\"Value(data={self.data:.4f}, grad={self.grad:.4f})\"\n\n# Test it!\na = Value(2.0, label='a')\nb = Value(3.0, label='b')\nc = a + b\nc.label = 'c'\n\nc.backward()  # Compute gradients!\n\nprint(\"After backward():\")\nprint(f\"a = {a}  (grad=1.0 because dc/da = 1)\")\nprint(f\"b = {b}  (grad=1.0 because dc/db = 1)\")\nprint(f\"c = {c}  (grad=1.0 because dc/dc = 1)\")"

In [ ]:
import numpy as np

# Input data (3 samples, 2 features) - regular floats
X = np.array([
    [1.0, 2.0],   # Sample 1: x1=1.0, x2=2.0
    [2.0, 3.0],   # Sample 2: x1=2.0, x2=3.0
    [3.0, 4.0]    # Sample 3: x1=3.0, x2=4.0
])

# Target values - regular floats
y_target = np.array([5.0, 8.0, 11.0])

print(\"Data:\")\nprint(f\"X shape: {X.shape}\")\nprint(f\"X:\\n{X}\")\nprint(f\"\\ny_target shape: {y_target.shape}\")\nprint(f\"y_target: {y_target}\")"

## Exercise 3: Matrix Multiplication with NumPy

### Problem

You're building a simple neural network layer. Given:
- Input: `X` with shape (batch_size=3, input_features=4)
- Weights: `W` with shape (input_features=4, output_features=2)
- Bias: `b` with shape (output_features=2,)

Compute the output: `Y = X @ W + b`

### Your Task

1. Use NumPy to perform matrix multiplication
2. Add the bias (broadcasting will handle the shape)
3. Verify the output shape is (3, 2)

**Hint:** Review the `np.matmul` documentation if needed!

### Solution

## Exercise 4: 2D Linear Regression with Gradient Calculation

### Problem Setup

You have:
- **Input data**: 3 samples with 2 features each (as regular floats/numpy arrays)
- **Weights**: w0 (bias), w1, w2 (as Value objects - these are learnable!)
- **Model**: `y_pred = w0 + w1*x1 + w2*x2`
- **Target**: y_target (ground truth)
- **Loss**: Mean Squared Error (MSE)

### Your Task

1. Initialize weights as Value objects: `[w0, w1, w2]`
2. Compute predictions for all 3 samples using the linear model
3. Compute MSE loss
4. Call `backward()` to compute gradients
5. Extract gradients into a shape-[3] array

**Important:**
- Weights (w0, w1, w2) must be Value objects
- Input data (x1, x2) are regular floats
- Use NumPy for data handling
- Use the Value class with backward pass you implemented in Exercise 2
- Final gradient should be shape [3] matching the 3 weights

### Given Data

## Summary

### What You've Learned

✅ **Computation graphs** track operations and their connections

✅ **Value class** records computation history for each operation

✅ **Forward propagation** computes outputs from inputs through the graph

✅ **Visualization** helps understand and debug computation graphs

✅ **Gradients** measure how sensitive outputs are to input changes

✅ **Numerical gradients** verify our understanding by nudging inputs

✅ **Chain rule** connects gradients through nested operations

✅ **NumPy fundamentals** for efficient array operations

✅ **Matrix multiplication** with `@` operator and `np.matmul`

### Why This Matters

You've built the complete foundation for automatic differentiation:
- ✓ Forward pass records operations (this lab)
- ✓ Gradient concepts and chain rule (this lab)
- ⏳ Automatic backward pass (next lab)

### Next Steps

In **Lab 3**, you'll:
- Build neural network components (Neuron, Layer, MLP)
- Implement forward propagation through networks
- **Implement backward propagation** for each operator (manual autograd)
- Understand how gradients flow automatically through the graph

**Great work!** 🎉 You now understand:
- How deep learning frameworks track computations
- What gradients are and why they matter
- The foundation for automatic differentiation

## Exercise 2: Matrix Multiplication with NumPy

### Problem

You're building a simple neural network layer. Given:
- Input: `X` with shape (batch_size=3, input_features=4)
- Weights: `W` with shape (input_features=4, output_features=2)
- Bias: `b` with shape (output_features=2,)

Compute the output: `Y = X @ W + b`

### Your Task

1. Use NumPy to perform matrix multiplication
2. Add the bias (broadcasting will handle the shape)
3. Verify the output shape is (3, 2)

**Hint:** Review the `np.matmul` documentation if needed!

In [ ]:
import numpy as np

# Given data (don't modify)
X = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

W = np.array([[0.1, 0.2],
              [0.3, 0.4],
              [0.5, 0.6],
              [0.7, 0.8]])

b = np.array([1.0, 2.0])

print(f"X shape: {X.shape}  (3 samples, 4 features each)")
print(f"W shape: {W.shape}  (4 input features, 2 output features)")
print(f"b shape: {b.shape}  (2 output features)\n")

# Y = ...

# Uncomment after implementing:
# print(f"Y shape: {Y.shape}  (should be (3, 2))")
# print(f"Y:\n{Y}")

# Verify
# expected_shape = (3, 2)
# assert Y.shape == expected_shape, f"Expected shape {expected_shape}, got {Y.shape}"
# print("\n✓ Matrix multiplication: PASS")

### Solution

## Summary

### What You've Learned

✅ **Computation graphs** track operations and their connections

✅ **Value class** records computation history for each operation

✅ **Forward propagation** computes outputs from inputs through the graph

✅ **Visualization** helps understand and debug computation graphs

✅ **NumPy fundamentals** for efficient array operations

✅ **Matrix multiplication** with `@` operator and `np.matmul`

### Why This Matters

You've built the foundation for automatic differentiation:
- ✓ Forward pass records operations (this lab)
- ⏳ Backward pass computes gradients (next lab)

### Next Steps

In **Lab 3**, you'll:
- Build neural network components (Neuron, Layer, MLP)
- Implement forward propagation through networks
- Implement backward propagation manually
- Understand how gradients flow through the graph

**Great work!** 🎉 You now understand how deep learning frameworks track computations.